# 06 — 2013 train-only normalization

This notebook calculates permanent DBZ/VEL channel
normalization statistics for the first 2013 baseline.

Only the deterministic internal training split is used.
Validation and official test frames are excluded.

Finite values contribute to mean and standard deviation.
During modeling, non-finite values will be mapped to zero
after z-score normalization, corresponding to train-mean
imputation.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.6"
VALIDATION_FRACTION = 0.20
VALIDATION_SEED = 20260913

BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)

EXPERIMENT_DIRECTORY = (
    BACKUP_ROOT
    / "experiments"
    / "2013_baseline_v1"
)
NORMALIZATION_PATH = (
    EXPERIMENT_DIRECTORY
    / "normalization.json"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_normalization"
)

for required_path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required path: "
            f"{required_path}"
        )

print("package:", PACKAGE_PATH)
print("archive:", DRIVE_ARCHIVE_PATH)
print(
    "normalization output:",
    NORMALIZATION_PATH,
)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl
archive: /content/drive/MyDrive/TorNet_Backup/tornet_2013.tar.gz
normalization output: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1/normalization.json


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl'], returncode=0)

In [4]:
import tornado_detection

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

print(
    "tornado_detection:",
    tornado_detection.__version__,
)


tornado_detection: 0.1.6


In [5]:
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
)

canonical_index = (
    load_canonical_frame_index(
        MANIFESTS_ROOT
    )
)
assigned_index = assign_model_splits(
    canonical_index,
    validation_fraction=(
        VALIDATION_FRACTION
    ),
    seed=VALIDATION_SEED,
)

training_index = (
    assigned_index.loc[
        assigned_index["year"].eq(2013)
        & assigned_index[
            "model_split"
        ].eq("train")
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(training_index) == 11_056
assert (
    training_index["file_id"].nunique()
    == 2_764
)
assert training_index[
    "split"
].eq("train").all()
assert training_index[
    "model_split"
].eq("train").all()
assert int(
    training_index[
        "frame_label"
    ].sum()
) == 445

print(
    "training frames:",
    f"{len(training_index):,}",
)
print(
    "training files:",
    f"{training_index['file_id'].nunique():,}",
)
print(
    "positive frames:",
    int(
        training_index[
            "frame_label"
        ].sum()
    ),
)


training frames: 11,056
training files: 2,764
positive frames: 445


In [6]:
import shutil
import tarfile
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter()
    - copy_started
)

required_members = set(
    training_index[
        "archive_member"
    ].unique()
)
extracted_members = set()

extraction_started = (
    time.perf_counter()
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT
            / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing_members = (
    required_members
    - extracted_members
)

if missing_members:
    raise RuntimeError(
        "Missing training files: "
        f"{sorted(missing_members)[:10]}"
    )

assert len(extracted_members) == 2_764

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "extracted files:",
    f"{len(extracted_members):,}",
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)


copy seconds: 62.665
extracted files: 2,764
extraction seconds: 33.206


In [7]:
import numpy as np
import xarray as xr

from tornado_detection.data import (
    build_frame_tensor,
)

channel_names = [
    "DBZ_sweep_0",
    "DBZ_sweep_1",
    "VEL_sweep_0",
    "VEL_sweep_1",
]
channel_count = len(channel_names)

finite_counts = np.zeros(
    channel_count,
    dtype=np.int64,
)
finite_sums = np.zeros(
    channel_count,
    dtype=np.float64,
)
finite_sum_squares = np.zeros(
    channel_count,
    dtype=np.float64,
)

frames_processed = 0
label_mismatches = []

scan_started = time.perf_counter()

for (
    archive_member,
    member_frames,
) in training_index.groupby(
    "archive_member",
    sort=False,
):
    path = (
        EXTRACTION_ROOT
        / archive_member
    )

    with xr.open_dataset(
        path,
        engine="netcdf4",
    ) as dataset:
        for row in (
            member_frames.itertuples(
                index=False
            )
        ):
            result = build_frame_tensor(
                dataset,
                int(row.frame_index),
            )

            if (
                result.label
                != int(row.frame_label)
            ):
                label_mismatches.append(
                    row.frame_id
                )

            values = result.values

            for channel in range(
                channel_count
            ):
                channel_values = (
                    values[
                        :,
                        :,
                        channel,
                    ]
                )
                finite_mask = np.isfinite(
                    channel_values
                )
                finite = channel_values[
                    finite_mask
                ].astype(
                    np.float64,
                    copy=False,
                )

                finite_counts[channel] += (
                    finite.size
                )
                finite_sums[channel] += (
                    finite.sum(
                        dtype=np.float64
                    )
                )
                finite_sum_squares[
                    channel
                ] += np.square(
                    finite,
                    dtype=np.float64,
                ).sum(
                    dtype=np.float64
                )

            frames_processed += 1

scan_seconds = (
    time.perf_counter()
    - scan_started
)

if label_mismatches:
    raise AssertionError(
        "Manifest/NetCDF label mismatches: "
        f"{label_mismatches[:10]}"
    )

assert frames_processed == 11_056
assert np.all(finite_counts > 0)

channel_means = (
    finite_sums
    / finite_counts
)
channel_variances = (
    finite_sum_squares
    / finite_counts
    - np.square(channel_means)
)
channel_variances = np.maximum(
    channel_variances,
    0.0,
)
channel_stds = np.sqrt(
    channel_variances
)

if not np.isfinite(
    channel_means
).all():
    raise AssertionError(
        "Non-finite channel means"
    )

if (
    not np.isfinite(
        channel_stds
    ).all()
    or np.any(channel_stds <= 0)
):
    raise AssertionError(
        "Invalid channel standard deviations"
    )

possible_values_per_channel = (
    frames_processed * 120 * 240
)
finite_fractions = (
    finite_counts
    / possible_values_per_channel
)

print(
    "frames processed:",
    f"{frames_processed:,}",
)
print(
    "label mismatches:",
    len(label_mismatches),
)
print(
    "scan seconds:",
    round(scan_seconds, 3),
)
print(
    "frames/s:",
    round(
        frames_processed
        / scan_seconds,
        3,
    ),
)

for index, name in enumerate(
    channel_names
):
    print()
    print(name)
    print(
        "  finite count:",
        int(finite_counts[index]),
    )
    print(
        "  finite fraction:",
        round(
            float(
                finite_fractions[
                    index
                ]
            ),
            8,
        ),
    )
    print(
        "  mean:",
        float(
            channel_means[index]
        ),
    )
    print(
        "  std:",
        float(
            channel_stds[index]
        ),
    )


frames processed: 11,056
label mismatches: 0
scan seconds: 138.188
frames/s: 80.007

DBZ_sweep_0
  finite count: 197613717
  finite fraction: 0.62062115
  mean: 23.44608928387294
  std: 14.114453997333369

DBZ_sweep_1
  finite count: 198288707
  finite fraction: 0.622741
  mean: 23.321601923098928
  std: 14.18105635281608

VEL_sweep_0
  finite count: 172772857
  finite fraction: 0.54260651
  mean: -2.4620917570547554
  std: 16.99755760657217

VEL_sweep_1
  finite count: 176580658
  finite fraction: 0.5545652
  mean: -2.225398534660008
  std: 17.793295326784364


In [8]:
import datetime
import json

normalization = {
    "artifact_kind": (
        "channel_normalization"
    ),
    "created_at_utc": (
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
    ),
    "package_version": PACKAGE_VERSION,
    "year": 2013,
    "official_source_split": "train",
    "model_split": "train",
    "validation_fraction": (
        VALIDATION_FRACTION
    ),
    "validation_seed": (
        VALIDATION_SEED
    ),
    "variables": [
        "DBZ",
        "VEL",
    ],
    "channel_order": channel_names,
    "tensor_layout": (
        "azimuth,range,channel"
    ),
    "tensor_shape": [
        120,
        240,
        4,
    ],
    "training_frame_count": int(
        frames_processed
    ),
    "training_file_count": int(
        training_index[
            "file_id"
        ].nunique()
    ),
    "training_positive_frame_count": (
        int(
            training_index[
                "frame_label"
            ].sum()
        )
    ),
    "finite_counts": [
        int(value)
        for value in finite_counts
    ],
    "finite_fractions": [
        float(value)
        for value in finite_fractions
    ],
    "means": [
        float(value)
        for value in channel_means
    ],
    "standard_deviations": [
        float(value)
        for value in channel_stds
    ],
    "nonfinite_policy": (
        "z-score using train-only finite "
        "mean/std, then map non-finite "
        "values to zero"
    ),
    "label_mismatch_count": int(
        len(label_mismatches)
    ),
}

EXPERIMENT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

if NORMALIZATION_PATH.exists():
    existing = json.loads(
        NORMALIZATION_PATH.read_text()
    )

    comparable_keys = [
        "package_version",
        "year",
        "official_source_split",
        "model_split",
        "validation_fraction",
        "validation_seed",
        "variables",
        "channel_order",
        "tensor_shape",
        "training_frame_count",
        "training_file_count",
        "training_positive_frame_count",
        "finite_counts",
        "means",
        "standard_deviations",
        "nonfinite_policy",
        "label_mismatch_count",
    ]

    differences = {
        key: {
            "existing": existing.get(
                key
            ),
            "current": normalization.get(
                key
            ),
        }
        for key in comparable_keys
        if existing.get(key)
        != normalization.get(key)
    }

    if differences:
        raise RuntimeError(
            "Existing normalization artifact "
            "differs from this run: "
            f"{differences}"
        )

    print(
        "Existing normalization artifact "
        "matches this run"
    )
else:
    NORMALIZATION_PATH.write_text(
        json.dumps(
            normalization,
            indent=2,
            sort_keys=True,
        )
        + "\n"
    )
    print(
        "Wrote:",
        NORMALIZATION_PATH,
    )

print()
print(
    json.dumps(
        normalization,
        indent=2,
        sort_keys=True,
    )
)


Wrote: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1/normalization.json

{
  "artifact_kind": "channel_normalization",
  "channel_order": [
    "DBZ_sweep_0",
    "DBZ_sweep_1",
    "VEL_sweep_0",
    "VEL_sweep_1"
  ],
  "created_at_utc": "2026-09-13T19:11:48.847792+00:00",
  "finite_counts": [
    197613717,
    198288707,
    172772857,
    176580658
  ],
  "finite_fractions": [
    0.620621146511698,
    0.6227410047586027,
    0.5426065063967278,
    0.5545651996402154
  ],
  "label_mismatch_count": 0,
  "means": [
    23.44608928387294,
    23.321601923098928,
    -2.4620917570547554,
    -2.225398534660008
  ],
  "model_split": "train",
  "nonfinite_policy": "z-score using train-only finite mean/std, then map non-finite values to zero",
  "official_source_split": "train",
  "package_version": "0.1.6",
  "standard_deviations": [
    14.114453997333369,
    14.18105635281608,
    16.99755760657217,
    17.793295326784364
  ],
  "tensor_layout": "azimuth,range,c

In [9]:
shutil.rmtree(EXTRACTION_ROOT)
LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()
assert NORMALIZATION_PATH.is_file()

print(
    "Removed all Colab-local normalization "
    "artifacts"
)
print(
    "Preserved normalization:",
    NORMALIZATION_PATH,
)


Removed all Colab-local normalization artifacts
Preserved normalization: /content/drive/MyDrive/TorNet_Backup/experiments/2013_baseline_v1/normalization.json
